# CIFAR-10 + SVP Pruning (ResNet-56)

Notebook này thực hiện quy trình **SVP (Singular Value Pruning)** cho mô hình **ResNet-56** trên tập dữ liệu **CIFAR-10**.

**Quy trình:**
1. **Load Dense Model**: Sử dụng ResNet-56 (pretrained hoặc train mới).
2. **SVP-GAM Analysis**: Phân tích singular values để tìm tỷ lệ pruning tối ưu cho từng layer dựa trên ngân sách (Budget).
3. **Structured Pruning**: Cắt tỉa các filter ít quan trọng nhất sử dụng thuật toán GEM (Nuclear Norm).
4. **Finetune**: Huấn luyện lại mô hình sau khi prune để phục hồi độ chính xác.
5. **Evaluation**: So sánh Accuracy, Params và FLOPs giữa model gốc và model đã prune.


## Cấu hình


In [ ]:
!pip install python-dotenv wandb tensorly ruptures thop

In [ ]:
import os
import sys
from pathlib import Path
try:
    from dotenv import load_dotenv
    import wandb
except ImportError:
    
    from dotenv import load_dotenv
    import wandb

# --- Cấu hình đường dẫn ---
REPO_ROOT = Path("/kaggle/input/datasets/chvminh/onestagecode")
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
# Load biến môi trường từ .env
load_dotenv(REPO_ROOT / ".env")
# Thêm SVP_main vào path để có thể import data, models trực tiếp
SVP_MAIN_PATH = REPO_ROOT / "SVP_main"
if str(SVP_MAIN_PATH) not in sys.path:
    sys.path.insert(0, str(SVP_MAIN_PATH))

import torch
import torch.nn as nn
import numpy as np

# Thêm try-except để báo lỗi rõ ràng nếu thiếu lib
try:
    from SVP_main.models_svp.resnet import resnet56
    from SVP_main.data import get_dataloaders
    print("Successfully imported models and data loaders from SVP_main.")
except ImportError as e:
    print(f"Error importing from SVP_main: {e}")
    print("Hãy đảm bảo thư mục 'SVP_main' tồn tại và chứa các file 'models_svp/resnet.py' và 'data.py'.")
    # Tái định nghĩa các hàm giả lập để notebook không bị crash ở các cell sau nếu chưa có code, 
    # nhưng ở đây chúng ta muốn nó fail rõ ràng để người dùng biết.
    raise e

from pruning.svp import SVPPruner
from pruning.norms import get_weight
try:
    import tensorly
    tensorly.set_backend("pytorch")
except ImportError:
    pass

# --- Thông số chạy (Theo bài báo SVP) ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "./data" # Đường dẫn lưu CIFAR10
BATCH_SIZE = 128
TARGET_RATE = 0.5
BUDGET = "flops"
EPOCHS = 300        
LR = 0.1            
MOMENTUM = 0.9
WEIGHT_DECAY = 0.005

print(f"Device: {DEVICE}")
print(f"Repo root: {REPO_ROOT}")

# Khởi tạo WandB
if os.getenv("WANDB_API_KEY"):
    wandb.login(key=os.getenv("WANDB_API_KEY"))
    wandb.init(
        project=os.getenv("WANDB_PROJECT", "SVP-CIFAR10"),
        config={
            "model": "resnet56",
            "dataset": "cifar10",
            "target_rate": TARGET_RATE,
            "budget": BUDGET,
            "epochs": EPOCHS,
            "lr": LR
        }
    )


## 1. Chuẩn bị Dữ liệu & Mô hình


In [ ]:
# Load CIFAR10
train_loader, test_loader = get_dataloaders(
    num_classes=10,
    dataset_path=DATA_PATH,
    mixup_alpha=0.2,
    cutmix_alpha=1.0,
    batch_size=BATCH_SIZE
)
model = resnet56(compress_rate=[0.0]*100, num_classes=10).to(DEVICE)

print("Model ResNet-56 initialized.")


## 2. Phân tích SVP-GAM

Để áp dụng `SVPPruner` từ repo này lên mô hình ResNet-56 của `SVP-main`, chúng ta cần một wrapper vì kiến trúc ResNet-CIFAR khác với YOLOv1.


In [ ]:
class CIFARModelScope:
    def __init__(self, model):
        self.model = model
    
    def get_prunable_layers(self, pruning_type="structured"):
        # Lấy tất cả các layer Conv2d trong ResNet
        # Trong ResNet-CIFAR, chúng ta thường prune các conv trong BasicBlock
        layers = []
        for name, m in self.model.named_modules():
            if isinstance(m, nn.Conv2d):
                # Tạo một wrapper giả lập PrunableConv để dùng được với SVPPruner
                # Hoặc đơn giản là trả về module đó nếu SVPPruner hỗ trợ
                layers.append(m)
        return layers

# Patch SVPPruner._get_layers để hoạt động với list layer trực tiếp
# (Hoặc sử dụng logic tùy biến)
pruner = SVPPruner(CIFARModelScope(model), input_size=32)

# Override lại method _get_layers của pruner để nó lấy đúng layer của ResNet56
def resnet_get_layers():
    layers = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            # Giả lập structure mà SVPPruner mong đợi (cần có .conv)
            class LayerWrapper:
                def __init__(self, conv): self.conv = conv
            layers.append(LayerWrapper(m))
    return layers

pruner._get_layers = resnet_get_layers

print("Analyzing singular values...")
layers, channels_to_keep, compress_rates = pruner.compute_allocation(
    target_rate=TARGET_RATE, 
    use_flops=(BUDGET == "flops")
)

print("\nTop 5 layers allocation:")
for i in range(min(5, len(compress_rates))):
    print(f"Layer {i}: Keep {channels_to_keep[i]} filters (Prune {compress_rates[i]*100:.1f}%)")


## 3. Structured Pruning (GEM)

Sử dụng Nuclear Norm để chọn ra các filter quan trọng nhất cho mỗi layer.


In [ ]:
@torch.no_grad()
def apply_svp_pruning(model, pruner, channels_to_keep):
    layers = pruner._get_layers()
    masks = []
    for i, layer in enumerate(layers):
        weight = layer.conv.weight.data
        n_keep = channels_to_keep[i]
        
        # Sử dụng GEM logic từ svp.py
        from pruning.svp import select_filters_gem_by_nuclear_norm
        mask = select_filters_gem_by_nuclear_norm(weight, n_keep)
        
        # Áp dụng mask (zero out pruned filters)
        mask_v = mask.view(-1, 1, 1, 1)
        layer.conv.weight.data.mul_(mask_v)
        if layer.conv.bias is not None:
            layer.conv.bias.data.mul_(mask)
            
        masks.append(mask)
    return masks

print("Applying GEM pruning...")
masks = apply_svp_pruning(model, pruner, channels_to_keep)
print("Pruning finished. Weights zeroed out.")


# --- Finetune & Recovery ---

# Hyperparameters theo bài báo: SGD, LR 0.1, Scheduler giảm dần



In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300)

criterion = nn.CrossEntropyLoss()

def train(model, loader, optimizer, criterion, epoch):
    model.train()
    total_loss = 0
    correct = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # Quan trọng: Giữ các trọng số đã prune luôn bằng 0 trong quá trình train
        for i, m in enumerate(resnet_get_layers()):
            mask = masks[i].view(-1, 1, 1, 1)
            if m.conv.weight.grad is not None:
                m.conv.weight.grad.data.mul_(mask)
            # Đồng thời đảm bảo trọng số thực tế không bị trôi do weight decay hoặc các cập nhật khác
            m.conv.weight.data.mul_(mask)
            if m.conv.bias is not None:
                m.conv.bias.data.mul_(masks[i])
            
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        if targets.ndim > 1:
            _, targets_idx = targets.max(1)
        else:
            targets_idx = targets
        correct += predicted.eq(targets_idx).sum().item()
    
    avg_loss = total_loss / len(loader)
    avg_acc = correct / len(loader.dataset)
    
    # Log to WandB
    if wandb.run:
        wandb.log({
            "train/loss": avg_loss,
            "train/acc": avg_acc,
            "train/lr": optimizer.param_groups[0]['lr'],
            "epoch": epoch
        })
    
    return avg_loss, avg_acc

print(f"Starting finetune for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    loss, acc = train(model, train_loader, optimizer, criterion, epoch)
    scheduler.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss:.4f} - Acc: {acc*100:.2f}%")

## 5. Evaluation


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            correct += predicted.eq(targets).sum().item()
    return correct / len(loader.dataset)

final_acc = evaluate(model, test_loader)
print(f"Final Accuracy on Test Set: {final_acc*100:.2f}%")
if wandb.run:
    wandb.log({"test/acc": final_acc})
    wandb.finish()

# Tính toán Params thực tế (không tính trọng số zero)
total_params = 0
active_params = 0
for m in model.modules():
    if isinstance(m, nn.Conv2d):
        total_params += m.weight.numel()
        # Đếm số filter có norm > 0
        norms = torch.norm(m.weight.data.view(m.weight.size(0), -1), dim=1)
        active_filters = (norms > 1e-6).sum().item()
        active_params += active_filters * m.weight.shape[1] * m.weight.shape[2] * m.weight.shape[3]

print(f"Total Conv Params: {total_params/1e6:.2f}M")
print(f"Active Conv Params: {active_params/1e6:.2f}M (Ratio: {active_params/total_params*100:.1f}%)")